# Polyphase Resampler Lab

Keep `polyphase_lab.py` in the same folder as this notebook. Needs `numpy`, `scipy`, `matplotlib`, and `ipywidgets` for the interactive cell.

In [ ]:
# %pip install numpy scipy matplotlib ipywidgets   # uncomment if anything is missing
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import polyphase_lab as pl

## 1. Interactive lab

Everything redraws when you release a slider. Step **Output index m** to watch the commutator move and new samples shift in.

In [ ]:
pl.lab()

## 2. The same pieces as plain function calls

Use these when you want to script something or keep a figure.

In [ ]:
L, M = 3, 2
h = pl.design_prototype(L, M, N=48, method='equiripple', pass_edge=0.8, stop_edge=1.2, weight=1)
print(pl.filter_stats(h, L, M))
pl.plot_filter(h, L, M, m=23);

In [ ]:
pl.plot_structure(h, L, M, m=23, form='L');
pl.plot_structure(h, L, M, m=23, form='M');

In [ ]:
# The raw table behind the diagram: x index held in each unit, tap index that weights it
x_idx, tap_idx, active = pl.leg_contents(h, L, M, m=23, form='L')
print('active leg:', active)
print('leg', active, 'holds x[n] for n =', x_idx[active])
print('weighted by h[i] for i =', tap_idx[active])

In [ ]:
pl.plot_legs(h, L, M, m=23);

In [ ]:
pl.plot_spectra(h, L, M, signal='band', bw=0.75);

## 3. Compare the three designs

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for name, kw in {'equiripple': dict(method='equiripple'), 'least-squares': dict(method='ls'),
                 'half-band (L=2)': dict(method='halfband')}.items():
    Lx = 2 if 'half' in name else L
    hh = pl.design_prototype(Lx, 1, 47, **kw)
    w, H = signal.freqz(hh, worN=4096)
    ax.plot(w / np.pi, 20 * np.log10(np.abs(H) + 1e-12), label=f'{name}, {len(hh)} taps')
ax.set(ylim=(-120, 5), xlabel='frequency ( x prototype Nyquist )', ylabel='dB'); ax.grid(True); ax.legend();

## 4. Run real samples through the structure

`polyphase_resample` computes each output exactly the way the L-leg diagram shows it. It should match SciPy's `upfirdn`.

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(2000)
y = pl.polyphase_resample(x, h, L, M)
ref = signal.upfirdn(L * h, x, L, M)[:len(y)]
print('max difference vs scipy.signal.upfirdn:', np.abs(y - ref).max())